In [ ]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
config_path = os.environ.get("DATUM_API_CONFIG_PATH", "../ops/datum_api_config.json")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [ ]:
# Import basic modules
import pandas as pd
from datum_api_client import DatumApi
import datetime
from datetime import timedelta
from typing import Optional, List, Dict, Any


# Import warnings
import warnings
warnings.filterwarnings("ignore")
# pip install xlrd
# pip install openpyxl

In [ ]:
from __future__ import annotations


def daytwo_stats_v2_exporter(
    input_path: str,
    *,
    output_onefile_jsonl: str = "DAYTWO/onefile.jsonl",
    output_summary_csv: str = "DAYTWO/summary.csv",
    output_best_params_jsonl: str = "DAYTWO/best_params.jsonl",
    # raw per-(ticker,session) snapshot + exit move at each class — NOT aggregated into bins.
    # Needed for day-accurate replay (e.g. a Scanner "SNAPSHOT" view for one specific date):
    # summary.csv/onefile.jsonl only carry all-history bin aggregates, so neither can answer
    # "what actually happened on 2026-07-14" — only this file can.
    output_events_jsonl: str = "DAYTWO/events.jsonl",
    # SIGNAL: the snapshot the whole rating is built on (15:40). Nearest row to the target,
    # searched from BOTH sides within +/- signal_window_minutes.
    signal_hm: tuple = (15, 40),
    signal_window_minutes: int = 5,
    # ENTRY: where the position is actually opened (16:00), 20 minutes AFTER the signal.
    # This is the baseline the move is measured from — see move_from.
    entry_hm: tuple = (16, 0),
    entry_window_minutes: int = 5,
    # EXIT classes. BLUE2 (00:00) and BLUE3 (04:00) belong to the SAME session as the 15:40
    # signal — see session_rollover_min below for how the day boundary is defined.
    exit_hm: dict = None,   # {"POST1":(18,0), "POST2":(19,30), "BLUE1":(21,0), "BLUE2":(0,0), "BLUE3":(4,0)}
    exit_window_minutes: int = 5,
    # per-class widening, e.g. {"BLUE2": 15} if overnight bars are sparser than intraday ones
    exit_window_overrides: dict = None,
    # "entry"  -> move = Stack%_exit - Stack%_16:00  (what the trade actually earns)
    # "signal" -> move = Stack%_exit - Stack%_15:40  (also swallows the 15:40->16:00 drift)
    move_from: str = "entry",
    # dead zone: |move| <= move_threshold is not a tradable signal, so the observation is
    # DROPPED entirely and never reaches any bin. Consequence: total == long + short, and
    # long_rate reads as P(up | the move was decisive) — flat days are not in the sample.
    move_threshold: float = 0.6,
    # SESSION DAY: minutes-since-midnight BELOW this value belong to the previous session
    # day, so the whole overnight block (and the 00:00 BLUE2 / 04:00 BLUE3 exits in
    # particular) stays attached to the session that started at 15:40 on the previous
    # calendar date. Without this the calendar-date rollover at midnight would silently
    # drop every overnight exit.
    #
    # 300 = 05:00, deliberately NOT 04:00: the boundary must sit strictly after the LAST
    # exit target plus its window, otherwise the 04:00 BLUE3 rows get re-dated into the next
    # session and the class comes out empty. The 04:00-05:00 early pre-market hour is
    # therefore attached to the previous session, which nothing in this strategy reads.
    session_rollover_min: int = 300,
    # bins (signed — negative and positive values are separate bins)
    stack_bin_min: float = -53.0,
    stack_bin_max: float = 53.0,
    stack_bin_step: float = 3.0,
    bench_bin_min: float = -53.0,
    bench_bin_max: float = 53.0,
    bench_bin_step: float = 3.0,
    devsig_bin_min: float = -20.0,
    devsig_bin_max: float = 20.0,
    devsig_bin_step: float = 0.4,
    # best params selection
    best_min_rate: float = 0.60,
    best_min_total: int = 10,
    # global filter
    min_events_per_ticker: int = 1,
    # column names (all three are per-row, time-varying snapshot values)
    STOCK_NUM_FIELD: str = "Stack%",
    BENCH_NUM_FIELD: str = "Bench%",
    DEVSIG_FIELD: str = "dev_sig",   # the column in final.parquet is dev_sig, not DevSig
    # ADVANCED: the real signal stays anchored at signal_hm (15:40) — ADVANCED only pools
    # EXTRA historical (signal, entry, exit) observations from every hourly checkpoint of
    # the session into a SEPARATE, much larger bin set, and picks its own best_params from
    # that pooled dataset. The "standard" 15:40-only best_params is always computed too and
    # is never replaced by ADVANCED.
    #
    # Unlike OpenDoor — where the advanced offsets had to be spelled out by hand because the
    # "10m"/"30m" class names were minutes-after-market-open rather than minutes-after-entry
    # — here every offset is DERIVED from the real schedule, so the pooled observations keep
    # exactly the same signal->entry (20m) and signal->exit gaps as the live strategy:
    #   H:00 -> signal, H:20 -> entry, H:00+gap(class) -> exit.
    enable_advanced: bool = True,
    advanced_hours: Optional[List[int]] = None,  # None = every hour that has data
    # reading
    assume_sorted: bool = True,
    parquet_use_pyarrow: bool = True,
    csv_chunksize: int = 500_000,
    log_every_n_chunks: int = 5,
):
    """
    DayTwo v2 — same machinery as OpenDoor, but for the afternoon/overnight leg.

    SIGNAL (per ticker, per session):
      - Row CLOSEST to signal_hm (default 15:40), searched from both sides within
        +/- signal_window_minutes.
      - Capture 3 factors from that single snapshot: Stack% (ticker move), Bench% (market
        move), DevSig (deviation). These three, and only these, are what gets binned.

    ENTRY (per ticker, per session):
      - Row CLOSEST to entry_hm (default 16:00), same nearest-match rule.
      - The position is opened here, 20 minutes after the signal, so with move_from="entry"
        this Stack% is the baseline every exit is measured against. The 15:40 -> 16:00 drift
        is therefore NOT counted as profit; it is still exported per day as
        "drift_signal_to_entry" in events.jsonl so it can be inspected separately.
      - A session with no signal row OR no entry row produces no event at all.

    EXIT (per ticker, per session): five classes, each the nearest row within its window
      POST1 = 18:00, POST2 = 19:30, BLUE1 = 21:00, BLUE2 = 00:00, BLUE3 = 04:00
      (the last two sit on the next calendar date but inside the same session).
      - move = Stack%_exit - Stack%_baseline -> "long" if move > move_threshold,
        "short" if move < -move_threshold. |move| <= move_threshold is a dead zone: the
        observation is dropped and counted nowhere (so total == long + short).

    SESSION DAY: everything before session_rollover_min (05:00) is folded back into the
    previous calendar date, and time is handled in "session minutes" (00:00 -> 1440), so
    the whole 15:40 -> 00:00 span is one monotonically increasing timeline.

    RATING per (parameter in {stack, devsig, bench}) x (class) x (bin):
      - total, up, down
      - long_rate = up/total, short_rate = down/total, long_short_ratio = up/down (if down>0)
      - avg_long_move  = mean(move | move >  move_threshold in this bin)
      - avg_short_move = mean(move | move < -move_threshold in this bin)

    Bins are signed floor-bins: stack/bench step=3.0 over [-53, 53] (36 bins, -54.0..51.0),
    devsig step=0.4 over [-20, 20] (101 bins, -20.0..20.0) — all configurable. A bin label is
    its LEFT edge formatted "%.1f", so a step needing more than one decimal (e.g. 0.25) would
    collide labels and silently merge bins. Out-of-range values are clamped, which makes the
    two edge bins open-ended buckets rather than one-step-wide.

    best_params: per parameter x class x direction, bins with rate>=best_min_rate and
    total>=best_min_total are stitched into consecutive intervals, scored by
    rate*log1p(total), carrying weighted avg_long_move/avg_short_move through the merge.
    """
    import gc, json, time, math, gzip
    from collections import defaultdict
    from datetime import datetime
    from typing import Optional, List
    import numpy as np
    import pandas as pd
    from pathlib import Path

    if exit_hm is None:
        exit_hm = {"POST1": (18, 0), "POST2": (19, 30), "BLUE1": (21, 0),
                   "BLUE2": (0, 0), "BLUE3": (4, 0)}
    if exit_window_overrides is None:
        exit_window_overrides = {}
    if move_from not in ("entry", "signal"):
        raise ValueError("move_from must be 'entry' or 'signal'")

    CLASSES = list(exit_hm.keys())
    PARAMS = ("stack", "devsig", "bench")
    DAY_MIN = 24 * 60

    def _to_smin(h, m):
        # session minutes: anything before the rollover is "tomorrow morning" of the SAME
        # session, so it sorts after 23:59 instead of wrapping back to 0.
        t = h * 60 + m
        return t if t >= session_rollover_min else t + DAY_MIN

    signal_smin = _to_smin(*signal_hm)
    entry_smin  = _to_smin(*entry_hm)
    exit_smin   = {c: _to_smin(*t) for c, t in exit_hm.items()}
    exit_win    = {c: int(exit_window_overrides.get(c, exit_window_minutes)) for c in CLASSES}

    if entry_smin <= signal_smin:
        raise ValueError(f"entry_hm {entry_hm} must be after signal_hm {signal_hm}")
    _late = [c for c, s in exit_smin.items() if s <= entry_smin]
    if _late:
        raise ValueError(
            f"exit classes {_late} land before entry_hm {entry_hm} on the session timeline — "
            f"an overnight/early-morning exit requires session_rollover_min (now "
            f"{session_rollover_min}) to be set AFTER it, e.g. 300 (05:00) for a 04:00 exit"
        )
    # The nearest-match window must not spill past the session boundary: the half of it that
    # lands on the other side gets re-dated into the next session and can never match, which
    # would quietly halve (or empty) the class instead of failing.
    _spill = [c for c, s in exit_smin.items() if s + exit_win[c] >= session_rollover_min + DAY_MIN]
    if _spill:
        raise ValueError(
            f"exit window of {_spill} crosses the session boundary — raise "
            f"session_rollover_min (now {session_rollover_min}) above the last exit + window"
        )

    # gaps measured from the SIGNAL — these are what ADVANCED replays at every hourly checkpoint
    entry_gap = entry_smin - signal_smin
    exit_gap  = {c: s - signal_smin for c, s in exit_smin.items()}

    Path(output_onefile_jsonl).parent.mkdir(parents=True, exist_ok=True)
    Path(output_summary_csv).parent.mkdir(parents=True, exist_ok=True)
    Path(output_best_params_jsonl).parent.mkdir(parents=True, exist_ok=True)
    Path(output_events_jsonl).parent.mkdir(parents=True, exist_ok=True)

    def _open_gz(path, mode="wt"):
        if str(path).lower().endswith(".gz"):
            return gzip.open(path, mode, encoding="utf-8", newline="\n", compresslevel=6)
        return open(path, mode.replace("t", ""), encoding="utf-8", newline="\n")

    # lo/hi are the winning interval's bin boundaries — included so a live consumer can check
    # "does the ticker's CURRENT value fall in this good range" from summary.csv alone, without
    # a per-ticker onefile.jsonl fetch. avg_move is the winning interval's avg_long_move/
    # avg_short_move, for a live MINMOVE threshold check.
    BEST_FIELDS = ("rate", "total", "lo", "hi", "avg_move")
    # "adv_" columns mirror the standard ones exactly but are sourced from the hourly-pooled
    # ADVANCED bin set (best_params.advanced) instead of the 15:40-only standard set — only
    # present when enable_advanced=True.
    summary_cols = (
        ["ticker", "bench", "events_total", "days_with_entry", "advanced_events_total"] +
        [f"{p}_{c}_best_{d}_{f}" for p in PARAMS for c in CLASSES for d in ("long", "short") for f in BEST_FIELDS] +
        [f"adv_{p}_{c}_best_{d}_{f}" for p in PARAMS for c in CLASSES for d in ("long", "short") for f in BEST_FIELDS] +
        ["corr", "beta"]
    )
    pd.DataFrame(columns=summary_cols).to_csv(output_summary_csv, index=False, mode="w")

    onefile_f     = _open_gz(output_onefile_jsonl, "wt")
    best_params_f = _open_gz(output_best_params_jsonl, "wt")
    events_f      = _open_gz(output_events_jsonl, "wt")
    best_params_f.write(json.dumps({
        "meta": {"version": "daytwo_v2", "generated_at": datetime.utcnow().isoformat() + "Z"}
    }) + "\n")

    # ── helpers ──────────────────────────────────────────────────────────────
    def _js(x):
        if x is None: return None
        if isinstance(x, (np.floating, float)):
            return None if (np.isnan(x) or np.isinf(x)) else round(float(x), 6)
        if isinstance(x, (np.integer, int)): return int(x)
        if isinstance(x, (np.bool_, bool)): return bool(x)
        return x

    def _ok(x):
        try: return np.isfinite(float(x))
        except Exception: return False

    def _dstr(v):
        # session date is carried as a packed int (yyyymmdd) — formatting it per row would
        # cost a strftime over millions of rows, so it only happens when an event is written.
        v = int(v)
        return f"{v // 10000:04d}-{(v // 100) % 100:02d}-{v % 100:02d}"

    def _clamp(v, lo, hi): return max(lo, min(hi, v))

    def _sbin(v, lo, hi, step):
        # Signed floor-bin: value v falls into [floor(v/step)*step, +step). Negative and
        # positive values land in distinct bins on either side of 0.0 — no special-casing
        # needed since math.floor already handles the sign correctly.
        if not _ok(v): return None
        v = _clamp(float(v), lo, hi)
        return f"{round(math.floor(v / step) * step, 6):.1f}"

    def stack_bin(v):  return _sbin(v, stack_bin_min,  stack_bin_max,  stack_bin_step)
    def bench_bin(v):  return _sbin(v, bench_bin_min,  bench_bin_max,  bench_bin_step)
    def devsig_bin(v): return _sbin(v, devsig_bin_min, devsig_bin_max, devsig_bin_step)

    BIN_FN   = {"stack": stack_bin, "devsig": devsig_bin, "bench": bench_bin}
    BIN_STEP = {"stack": stack_bin_step, "devsig": devsig_bin_step, "bench": bench_bin_step}

    def _score(rate, total): return float(rate) * math.log1p(int(total))

    def _new_bin_stat():
        return {"long": 0, "short": 0, "total": 0, "long_sum": 0.0, "short_sum": 0.0}

    def _new_bin_store():
        # bin_store[param][cls][bin_label] -> stat dict
        return {p: {c: defaultdict(_new_bin_stat) for c in CLASSES} for p in PARAMS}

    def _accumulate_class(bin_store, cls, signal_vals, move):
        # dead zone -> neither long nor short, and not counted in total either
        if abs(move) <= move_threshold:
            return
        direction = "long" if move > 0 else "short"
        for p in PARAMS:
            b = BIN_FN[p](signal_vals.get(p))
            if b is None:
                continue
            st = bin_store[p][cls][b]
            st["total"] += 1
            st[direction] += 1
            if direction == "long":
                st["long_sum"] += move
            else:
                st["short_sum"] += move

    # ── per-ticker state ──────────────────────────────────────────────────────
    cur_ticker = None
    cur_day    = None          # packed session date (yyyymmdd)
    bench_seen = None
    static_set = False
    corr_s = beta_s = None

    # standard (15:40-anchored) per-session accumulators
    day_signal      = None     # {"stack":..,"devsig":..,"bench":..} snapshot at 15:40
    day_signal_dist = None     # |session minutes - signal target| of the held candidate
    day_entry_stack = None     # Stack% at 16:00 — the baseline moves are measured from
    day_entry_dist  = None
    day_exits       = {}       # cls -> Stack%_exit
    day_exit_dist   = {}       # cls -> |session minutes - class target|
    day_count       = 0

    # advanced (hourly-pooled) per-session accumulators, keyed by checkpoint session-minute
    adv_signal     = {}
    adv_entry      = {}
    adv_entry_dist = {}
    adv_exits      = {}
    adv_exit_dist  = {}

    bins_std = _new_bin_store()
    bins_adv = _new_bin_store() if enable_advanced else None
    adv_events_total = 0

    def _reset_ticker():
        nonlocal bench_seen, static_set, corr_s, beta_s
        nonlocal day_signal, day_signal_dist, day_entry_stack, day_entry_dist
        nonlocal day_exits, day_exit_dist, day_count
        nonlocal adv_signal, adv_entry, adv_entry_dist, adv_exits, adv_exit_dist
        nonlocal bins_std, bins_adv, adv_events_total
        bench_seen = None; static_set = False; corr_s = beta_s = None
        day_signal = None; day_signal_dist = None
        day_entry_stack = None; day_entry_dist = None
        day_exits = {}; day_exit_dist = {}; day_count = 0
        adv_signal = {}; adv_entry = {}; adv_entry_dist = {}; adv_exits = {}; adv_exit_dist = {}
        bins_std = _new_bin_store()
        bins_adv = _new_bin_store() if enable_advanced else None
        adv_events_total = 0

    def _reset_day():
        nonlocal day_signal, day_signal_dist, day_entry_stack, day_entry_dist
        nonlocal day_exits, day_exit_dist
        nonlocal adv_signal, adv_entry, adv_entry_dist, adv_exits, adv_exit_dist
        day_signal = None; day_signal_dist = None
        day_entry_stack = None; day_entry_dist = None
        day_exits = {}; day_exit_dist = {}
        adv_signal = {}; adv_entry = {}; adv_entry_dist = {}; adv_exits = {}; adv_exit_dist = {}

    def _finalize_day():
        nonlocal day_count, adv_events_total
        # both halves are required: the signal supplies the bins, the entry supplies the
        # baseline. A session missing either one is not a tradable observation.
        if day_signal is not None and day_entry_stack is not None:
            base = float(day_entry_stack) if move_from == "entry" else float(day_signal["stack"])
            day_count += 1
            event_row = {
                "ticker": cur_ticker,
                "date": _dstr(cur_day),
                "signal_stack": _js(day_signal["stack"]),
                "signal_devsig": _js(day_signal.get("devsig")),
                "signal_bench": _js(day_signal.get("bench")),
                "entry_stack": _js(day_entry_stack),
                "drift_signal_to_entry": _js(float(day_entry_stack) - float(day_signal["stack"])),
            }
            has_any_exit = False
            for c in CLASSES:
                exit_stack = day_exits.get(c)
                if exit_stack is None or not _ok(exit_stack):
                    event_row[f"move_{c}"] = None
                    continue
                move = float(exit_stack) - base
                # bins drop |move| <= move_threshold; events.jsonl keeps the RAW move so a
                # replay can re-derive direction under a different threshold.
                _accumulate_class(bins_std, c, day_signal, move)
                event_row[f"move_{c}"] = _js(move)
                has_any_exit = True
            if has_any_exit:
                events_f.write(json.dumps(event_row, ensure_ascii=False) + "\n")

        if enable_advanced:
            for c_min, sig in adv_signal.items():
                if advanced_hours is not None and ((c_min // 60) % 24) not in advanced_hours:
                    continue
                e_stack = adv_entry.get(c_min)
                if e_stack is None:
                    continue
                base = float(e_stack) if move_from == "entry" else float(sig["stack"])
                exits_c = adv_exits.get(c_min, {})
                hit = False
                for c in CLASSES:
                    exit_stack = exits_c.get(c)
                    if exit_stack is None or not _ok(exit_stack):
                        continue
                    _accumulate_class(bins_adv, c, sig, float(exit_stack) - base)
                    hit = True
                if hit:
                    # coverage counter: checkpoints that had signal+entry+at least one exit.
                    # Not equal to the sum of adv bin totals — dead-zone moves are excluded
                    # from the bins but the checkpoint still counts as observed.
                    adv_events_total += 1

    def _rating(st):
        tot = int(st["total"]); long_cnt = int(st["long"]); short_cnt = int(st["short"])
        return {
            "total": tot, "long": long_cnt, "short": short_cnt,
            "long_rate": round(long_cnt / tot, 4) if tot else None,
            "short_rate": round(short_cnt / tot, 4) if tot else None,
            "long_short_ratio": round(long_cnt / short_cnt, 4) if short_cnt > 0 else None,
            "avg_long_move": round(st["long_sum"] / long_cnt, 4) if long_cnt else None,
            "avg_short_move": round(st["short_sum"] / short_cnt, 4) if short_cnt else None,
        }

    def _best_for_param_class(bins_d, direction, step):
        # Consecutive-bin stitching: eligible neighbouring bins are merged into one interval,
        # carrying weighted avg_long_move/avg_short_move (via long_sum/short_sum) through.
        eligible = []
        for b_str, st in bins_d.items():
            tot = int(st["total"])
            if tot < best_min_total: continue
            cnt = int(st[direction])
            rate = cnt / tot if tot else 0.0
            if rate >= best_min_rate:
                try: eligible.append((float(b_str), b_str, dict(st)))
                except ValueError: pass
        eligible.sort(key=lambda x: x[0])
        if not eligible: return []

        intervals = []
        lo_s, hi_f, hi_s = eligible[0][1], eligible[0][0], eligible[0][1]
        agg = dict(eligible[0][2])

        for v, s, st in eligible[1:]:
            if abs(v - (hi_f + step)) < 1e-9:
                hi_f, hi_s = v, s
                for k in ("long", "short", "total", "long_sum", "short_sum"):
                    agg[k] += st[k]
            else:
                intervals.append((lo_s, hi_s, agg))
                lo_s, hi_f, hi_s = s, v, s
                agg = dict(st)
        intervals.append((lo_s, hi_s, agg))

        result = []
        for lo_s, hi_s, agg in intervals:
            r = _rating(agg)
            tot, cnt = r["total"], r[direction]
            rate = cnt / tot if tot else 0.0
            if tot >= best_min_total and rate >= best_min_rate:
                result.append({
                    "lo": lo_s, "hi": hi_s, "total": tot,
                    direction: cnt, "rate": round(rate, 4),
                    "avg_long_move": r["avg_long_move"], "avg_short_move": r["avg_short_move"],
                    "score": round(_score(rate, tot), 4),
                })
        result.sort(key=lambda x: x["score"], reverse=True)
        return result

    def _best_params_block(bin_store):
        best = {}
        for p in PARAMS:
            best[p] = {}
            for c in CLASSES:
                best[p][c] = {
                    "long":  _best_for_param_class(bin_store[p][c], "long",  BIN_STEP[p]),
                    "short": _best_for_param_class(bin_store[p][c], "short", BIN_STEP[p]),
                }
        return best

    # ── flush ticker ──────────────────────────────────────────────────────────
    def _flush():
        if cur_ticker is None:
            return
        _finalize_day()

        events_total = max(
            (int(sum(st["total"] for st in bins_std[p][c].values())) for p in PARAMS for c in CLASSES),
            default=0,
        )
        if events_total < min_events_per_ticker:
            _reset_ticker()
            return

        ratings_std = {p: {c: {b: _rating(st) for b, st in bins_std[p][c].items()} for c in CLASSES} for p in PARAMS}
        best_std = _best_params_block(bins_std)

        ratings_adv = None
        best_adv = None
        if enable_advanced:
            ratings_adv = {p: {c: {b: _rating(st) for b, st in bins_adv[p][c].items()} for c in CLASSES} for p in PARAMS}
            best_adv = _best_params_block(bins_adv)

        payload = {
            "ticker": cur_ticker,
            "bench": bench_seen,
            "static": {"corr": _js(corr_s), "beta": _js(beta_s)},
            "events_total": int(events_total),
            "days_with_entry": int(day_count),
            "advanced_events_total": int(adv_events_total) if enable_advanced else 0,
            "params": {
                "signal_hm": list(signal_hm),
                "signal_window_minutes": signal_window_minutes,
                "entry_hm": list(entry_hm),
                "entry_window_minutes": entry_window_minutes,
                "exit_hm": {c: list(t) for c, t in exit_hm.items()},
                "exit_window_minutes": {c: exit_win[c] for c in CLASSES},
                "move_from": move_from,
                "move_threshold": move_threshold,
                "session_rollover_min": session_rollover_min,
                "stack_bins":  {"min": stack_bin_min,  "max": stack_bin_max,  "step": stack_bin_step},
                "bench_bins":  {"min": bench_bin_min,  "max": bench_bin_max,  "step": bench_bin_step},
                "devsig_bins": {"min": devsig_bin_min, "max": devsig_bin_max, "step": devsig_bin_step},
                "best_min_rate": best_min_rate,
                "best_min_total": best_min_total,
                "enable_advanced": enable_advanced,
                "advanced_gaps": {"entry": entry_gap, **{f"exit_{c}": exit_gap[c] for c in CLASSES}} if enable_advanced else None,
            },
            "standard": {"ratings": ratings_std},
            "best_params": {"standard": best_std},
        }
        if enable_advanced:
            payload["advanced"] = {"ratings": ratings_adv}
            payload["best_params"]["advanced"] = best_adv

        onefile_f.write(json.dumps(payload, ensure_ascii=False) + "\n")

        row = {
            "ticker": cur_ticker, "bench": bench_seen,
            "events_total": int(events_total), "days_with_entry": int(day_count),
            "advanced_events_total": int(adv_events_total) if enable_advanced else 0,
        }
        for p in PARAMS:
            for c in CLASSES:
                long_best = best_std[p][c]["long"][0] if best_std[p][c]["long"] else None
                short_best = best_std[p][c]["short"][0] if best_std[p][c]["short"] else None
                row[f"{p}_{c}_best_long_rate"]     = _js(long_best["rate"]) if long_best else None
                row[f"{p}_{c}_best_long_total"]    = int(long_best["total"]) if long_best else None
                row[f"{p}_{c}_best_long_lo"]       = long_best["lo"] if long_best else None
                row[f"{p}_{c}_best_long_hi"]       = long_best["hi"] if long_best else None
                row[f"{p}_{c}_best_long_avg_move"] = _js(long_best["avg_long_move"]) if long_best else None
                row[f"{p}_{c}_best_short_rate"]     = _js(short_best["rate"]) if short_best else None
                row[f"{p}_{c}_best_short_total"]    = int(short_best["total"]) if short_best else None
                row[f"{p}_{c}_best_short_lo"]       = short_best["lo"] if short_best else None
                row[f"{p}_{c}_best_short_hi"]       = short_best["hi"] if short_best else None
                row[f"{p}_{c}_best_short_avg_move"] = _js(short_best["avg_short_move"]) if short_best else None
                if enable_advanced:
                    adv_long_best = best_adv[p][c]["long"][0] if best_adv[p][c]["long"] else None
                    adv_short_best = best_adv[p][c]["short"][0] if best_adv[p][c]["short"] else None
                    row[f"adv_{p}_{c}_best_long_rate"]     = _js(adv_long_best["rate"]) if adv_long_best else None
                    row[f"adv_{p}_{c}_best_long_total"]    = int(adv_long_best["total"]) if adv_long_best else None
                    row[f"adv_{p}_{c}_best_long_lo"]       = adv_long_best["lo"] if adv_long_best else None
                    row[f"adv_{p}_{c}_best_long_hi"]       = adv_long_best["hi"] if adv_long_best else None
                    row[f"adv_{p}_{c}_best_long_avg_move"] = _js(adv_long_best["avg_long_move"]) if adv_long_best else None
                    row[f"adv_{p}_{c}_best_short_rate"]     = _js(adv_short_best["rate"]) if adv_short_best else None
                    row[f"adv_{p}_{c}_best_short_total"]    = int(adv_short_best["total"]) if adv_short_best else None
                    row[f"adv_{p}_{c}_best_short_lo"]       = adv_short_best["lo"] if adv_short_best else None
                    row[f"adv_{p}_{c}_best_short_hi"]       = adv_short_best["hi"] if adv_short_best else None
                    row[f"adv_{p}_{c}_best_short_avg_move"] = _js(adv_short_best["avg_short_move"]) if adv_short_best else None
        row.update({"corr": _js(corr_s), "beta": _js(beta_s)})
        pd.DataFrame([row], columns=summary_cols).to_csv(output_summary_csv, mode="a", header=False, index=False)

        bp_row = {"ticker": cur_ticker, "bench": bench_seen, "best": {"standard": best_std}}
        if enable_advanced:
            bp_row["best"]["advanced"] = best_adv
        best_params_f.write(json.dumps(bp_row, ensure_ascii=False) + "\n")

        _reset_ticker()

    # ── chunk processor ───────────────────────────────────────────────────────
    def _process_chunk(chunk, ci):
        nonlocal cur_ticker, cur_day, bench_seen, static_set, corr_s, beta_s
        nonlocal day_signal, day_signal_dist, day_entry_stack, day_entry_dist

        req = {"ticker", "dt"}
        if not req.issubset(chunk.columns):
            raise KeyError(f"Missing columns: {sorted(req - set(chunk.columns))}")
        # _col() substitutes an all-NaN series for a column that is not there, which means a
        # misspelled field name does not fail — it silently empties that parameter's bins and
        # the run still "succeeds". Fail loudly instead.
        _num = {STOCK_NUM_FIELD, BENCH_NUM_FIELD, DEVSIG_FIELD} - set(chunk.columns)
        if _num:
            raise KeyError(
                f"numeric field(s) {sorted(_num)} not in the data — available: "
                f"{sorted(chunk.columns)}. A wrong name here would be silently read as NaN.")

        if not assume_sorted:
            chunk["dt"] = pd.to_datetime(chunk["dt"], errors="coerce", utc=True)
            chunk.sort_values(["ticker", "dt"], inplace=True)

        def _col(name):
            return chunk[name] if name in chunk.columns else pd.Series(np.nan, index=chunk.index)

        s_dt = pd.to_datetime(_col("dt"), errors="coerce", utc=True)
        ok   = s_dt.notna().to_numpy(copy=False)
        if not ok.any(): return

        s_dt2 = s_dt[ok]
        t_arr = (s_dt2.dt.hour.to_numpy(dtype="int32", copy=False) * 60 +
                 s_dt2.dt.minute.to_numpy(dtype="int32", copy=False))
        # session minutes + session date: shifting the timestamp back by the rollover makes
        # both fall out of the same subtraction, and keeps them monotonic across midnight.
        smin_arr = np.where(t_arr >= session_rollover_min, t_arr, t_arr + DAY_MIN).astype("int32")
        sess = s_dt2 - pd.Timedelta(minutes=session_rollover_min)
        sd_arr = (sess.dt.year.to_numpy(dtype="int32", copy=False) * 10000 +
                  sess.dt.month.to_numpy(dtype="int32", copy=False) * 100 +
                  sess.dt.day.to_numpy(dtype="int32", copy=False)).astype("int32")

        tk_arr = _col("ticker")[ok].to_numpy(copy=False)

        stock_arr  = pd.to_numeric(_col(STOCK_NUM_FIELD)[ok],  errors="coerce").to_numpy(dtype="float64", copy=False)
        bench_arr  = pd.to_numeric(_col(BENCH_NUM_FIELD)[ok],  errors="coerce").to_numpy(dtype="float64", copy=False)
        devsig_arr = pd.to_numeric(_col(DEVSIG_FIELD)[ok],     errors="coerce").to_numpy(dtype="float64", copy=False)

        bn_arr   = _col("bench")[ok].to_numpy(copy=False) if "bench" in chunk.columns else None
        corr_arr = _col("corr")[ok].to_numpy(copy=False)  if "corr"  in chunk.columns else None
        beta_arr = _col("beta")[ok].to_numpy(copy=False)  if "beta"  in chunk.columns else None

        for i in range(len(tk_arr)):
            tk = tk_arr[i]
            ds = int(sd_arr[i])
            smin = int(smin_arr[i])
            spct = float(stock_arr[i])
            bpct = float(bench_arr[i])
            dsig = float(devsig_arr[i])

            # ticker boundary
            if cur_ticker is not None and tk != cur_ticker:
                _flush()
                cur_ticker = tk; cur_day = ds
                _reset_day()

            if cur_ticker is None:
                cur_ticker = tk; cur_day = ds

            # static fields (bench label / corr / beta) — captured once per ticker
            if bn_arr is not None and bench_seen is None:
                v = bn_arr[i]
                if pd.notna(v) and str(v).strip():
                    bench_seen = str(v)

            if not static_set and corr_arr is not None and beta_arr is not None:
                c, b = corr_arr[i], beta_arr[i]
                if pd.notna(c) and pd.notna(b):
                    corr_s, beta_s = float(c), float(b)
                    static_set = True

            # session-day boundary
            if ds != cur_day:
                _finalize_day()
                cur_day = ds
                _reset_day()

            if not _ok(spct):
                continue

            # ── standard signal (15:40) / entry (16:00) / exits: nearest row to the target,
            # searched from BOTH sides within the class window ──
            d = abs(smin - signal_smin)
            if d <= signal_window_minutes and (day_signal_dist is None or d < day_signal_dist):
                day_signal = {
                    "stack": spct,
                    "devsig": dsig if _ok(dsig) else None,
                    "bench": bpct if _ok(bpct) else None,
                }
                day_signal_dist = d

            d = abs(smin - entry_smin)
            if d <= entry_window_minutes and (day_entry_dist is None or d < day_entry_dist):
                day_entry_stack = spct
                day_entry_dist = d

            for c, tgt in exit_smin.items():
                d = abs(smin - tgt)
                if d > exit_win[c]:
                    continue
                if day_exit_dist.get(c) is None or d < day_exit_dist[c]:
                    day_exits[c] = spct
                    day_exit_dist[c] = d

            # ── advanced: every H:00 checkpoint replays the same schedule ──
            if enable_advanced:
                if smin % 60 == 0:
                    adv_signal[smin] = {
                        "stack": spct,
                        "devsig": dsig if _ok(dsig) else None,
                        "bench": bpct if _ok(bpct) else None,
                    }

                # A row can serve checkpoint c_min only if |smin - (c_min + gap)| <= window,
                # and c_min is a multiple of 60 — so at most two checkpoints qualify and they
                # can be derived arithmetically instead of scanning every checkpoint per row.
                b0 = ((smin - entry_gap) // 60) * 60
                for c_min in (b0, b0 + 60):
                    if c_min not in adv_signal:
                        continue
                    d = abs(smin - (c_min + entry_gap))
                    if d > entry_window_minutes:
                        continue
                    if adv_entry_dist.get(c_min) is None or d < adv_entry_dist[c_min]:
                        adv_entry[c_min] = spct
                        adv_entry_dist[c_min] = d

                for c in CLASSES:
                    g = exit_gap[c]; w = exit_win[c]
                    b0 = ((smin - g) // 60) * 60
                    for c_min in (b0, b0 + 60):
                        if c_min not in adv_signal:
                            continue
                        d = abs(smin - (c_min + g))
                        if d > w:
                            continue
                        dists = adv_exit_dist.setdefault(c_min, {})
                        if dists.get(c) is None or d < dists[c]:
                            adv_exits.setdefault(c_min, {})[c] = spct
                            dists[c] = d

    # ── main read loop ────────────────────────────────────────────────────────
    t0 = time.time()
    total_rows = 0
    last_rows  = 0
    last_ts    = t0
    is_parquet = str(input_path).lower().endswith((".parquet", ".pq", ".parq"))

    print(f"START DayTwo v2  file={input_path}  parquet={is_parquet}")
    print(f"  signal={signal_hm} +/-{signal_window_minutes}m  entry={entry_hm} +/-{entry_window_minutes}m  move_from={move_from}")
    print(f"  exits={exit_hm}  windows={exit_win}")
    print(f"  move_threshold={move_threshold} (|move|<=thr dropped)  session_rollover={session_rollover_min}min")
    print(f"  min_events={min_events_per_ticker}  advanced={enable_advanced}")

    try:
        if is_parquet and parquet_use_pyarrow:
            import pyarrow.parquet as pq
            pf = pq.ParquetFile(input_path)
            wanted = ["ticker", "dt", "bench", "corr", "beta",
                      STOCK_NUM_FIELD, BENCH_NUM_FIELD, DEVSIG_FIELD]
            cols = [c for c in wanted if c in pf.schema.names]

            for ci in range(pf.num_row_groups):
                chunk = pf.read_row_group(ci, columns=cols).to_pandas()
                _process_chunk(chunk, ci + 1)
                total_rows += len(chunk)
                if (ci + 1) % log_every_n_chunks == 0:
                    now = time.time()
                    rps = (total_rows - last_rows) / max(now - last_ts, 1e-6)
                    print(f"[rg {ci+1:>4}/{pf.num_row_groups}] rows={total_rows:,} speed={rps:,.0f}/s elapsed={now-t0:.1f}s")
                    last_rows, last_ts = total_rows, now
                del chunk
                if (ci + 1) % 20 == 0: gc.collect()

        elif not is_parquet:
            for ci, chunk in enumerate(
                pd.read_csv(input_path, compression="infer", low_memory=False, chunksize=csv_chunksize), 1
            ):
                _process_chunk(chunk, ci)
                total_rows += len(chunk)
                if ci % log_every_n_chunks == 0:
                    now = time.time()
                    rps = (total_rows - last_rows) / max(now - last_ts, 1e-6)
                    print(f"[chunk {ci}] rows={total_rows:,} speed={rps:,.0f}/s elapsed={now-t0:.1f}s")
                    last_rows, last_ts = total_rows, now
                del chunk
                if ci % 20 == 0: gc.collect()

        else:
            df = pd.read_parquet(input_path)
            step = 1_000_000
            for ci, start in enumerate(range(0, len(df), step), 1):
                _process_chunk(df.iloc[start:start + step], ci)
                total_rows += len(df.iloc[start:start + step])

        _flush()
        elapsed = time.time() - t0
        print(f"DONE rows={total_rows:,} elapsed={elapsed:.1f}s")
        print(f"  onefile     = {output_onefile_jsonl}")
        print(f"  summary     = {output_summary_csv}")
        print(f"  best_params = {output_best_params_jsonl}")
        print(f"  events      = {output_events_jsonl}")

    finally:
        onefile_f.close()
        best_params_f.close()
        events_f.close()

In [ ]:
from pathlib import Path
import os


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env   = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path   = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve()   if sig_env   else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break

        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var (recommended).")

        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")

    return final_path, out_dir


# ── runner ────────────────────────────────────────────────────────────────────

FINAL_PATH, OUT_DIR = _resolve_orion_paths("daytwo")

daytwo_stats_v2_exporter(
    input_path=str(FINAL_PATH),
    output_onefile_jsonl=str(OUT_DIR / "onefile.jsonl.gz"),
    output_best_params_jsonl=str(OUT_DIR / "best_params.jsonl.gz"),
    output_summary_csv=str(OUT_DIR / "summary.csv"),
    output_events_jsonl=str(OUT_DIR / "events.jsonl.gz"),
    signal_hm=(15, 40), signal_window_minutes=5,
    entry_hm=(16, 0), entry_window_minutes=5,
    exit_hm={"POST1": (18, 0), "POST2": (19, 30), "BLUE1": (21, 0),
             "BLUE2": (0, 0), "BLUE3": (4, 0)},
    exit_window_minutes=5,
    # overnight bars are usually sparser than intraday ones — widen if BLUE* coverage is thin
    exit_window_overrides=None,
    move_from="entry",
    move_threshold=0.6,
    session_rollover_min=300,   # 05:00 — must stay after the 04:00 BLUE3 exit + its window
    stack_bin_min=-53.0, stack_bin_max=53.0, stack_bin_step=3.0,
    bench_bin_min=-53.0, bench_bin_max=53.0, bench_bin_step=3.0,
    devsig_bin_min=-20.0, devsig_bin_max=20.0, devsig_bin_step=0.4,
    best_min_rate=0.60, best_min_total=10, min_events_per_ticker=1,
    STOCK_NUM_FIELD="Stack%", BENCH_NUM_FIELD="Bench%", DEVSIG_FIELD="dev_sig",
    enable_advanced=True, advanced_hours=None,
    assume_sorted=True,
)
